In [ ]:
import pandas as pd
import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import seaborn as sns 
from scipy.stats import pearsonr
import json
from shapely.geometry import shape 
# from shapely.geometry import Polygon 
import json 
from shapely import wkt 
from shapely.geometry import Point
from pandas.tseries.offsets import Week
from statsmodels.tsa.stattools import kpss 
import statsmodels.api as sm
import warnings
import scipy.stats as stats
from scipy.stats import zscore
import contextily as ctx
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from shapely.ops import unary_union
from scipy.spatial import cKDTree
from pykrige.ok import OrdinaryKriging

In [ ]:
### Read hourly weather dataframe 
weather_df = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geo_Interpolation/Notebook/weather_concat.csv')

weather_df['Hour'] = pd.to_datetime(weather_df['Hour'])
weather_df = weather_df.rename(columns = {'Interpol_Precip':'Precip', 'Interpol_Temp':'Temp'})
weather_df = weather_df[['Hour', 'Precip', 'Temp', 'Location']]

In [ ]:
### Dictionary to store values of weather station names and data 

location_info = {
    'Location': ['Accra_Aca', 'G_Met', 'Temasco', 'St_Johns', 'Safisana', 'Nsawam', 'Agri_Impact', 'Accra_Girls', 'Legon', 'Madina', 'Berekuso'],
    'Coordinates': [
        (5.573103555, -0.244500082),
        (5.652019933, -0.16446233),
        (5.641413, -0.01187),
        (5.6383744, -0.2447151),
        (5.6836, -0.049468),
        (5.797283, -0.346325),
        (5.760172, -0.231223),
        (5.597071, -0.194248),
        (5.659788, -0.190434),
        (5.675314, -0.179166),
        (5.758029, -0.221716)
    ],
    'Elevation': [32.4, 71.6, 18.4, 46.0, 12.0, 62.0, 355.0, 63.0, 82.0, 60.4, 330.0]
}


location_df = pd.DataFrame(location_info)

# Split coordinates into Latitude and Longitude
location_df[['Latitude', 'Longitude']] = pd.DataFrame(location_df['Coordinates'].tolist(), index=location_df.index)
location_df.drop(columns='Coordinates', inplace=True)

# Merge with weather_df
weather_df = weather_df.merge(location_df, on='Location', how='left')

weather_df['geometry'] = weather_df.apply(
    lambda row: Point(row['Longitude'], row['Latitude']), axis=1
)

weather_gdf = gpd.GeoDataFrame(weather_df, geometry='geometry')
weather_gdf = weather_gdf.set_crs(epsg=4326)
weather_gdf = weather_gdf.to_crs('EPSG:32630')

### Separate into different gdfs based on location 

In [ ]:
location_vars = {
    'Accra_Aca': 'accra_aca_df_year',
    'G_Met': 'g_met_hq_df_year',
    'Temasco': 'temasco_df_year',
    'St_Johns': 'st_johns_df_year',
    'Safisana': 'safisana_df_year',
    'Nsawam': 'nsawam_df_year',
    'Agri_Impact': 'agri_impact_df_year',
    'Accra_Girls': 'accra_girls_df_year',
    'Legon': 'legon_df_year',
    'Madina': 'madina_df_year',
    'Berekuso': 'berekuso_df_year'
}

# Loop through and create separate GeoDataFrames
for loc, var_name in location_vars.items():
    globals()[var_name] = weather_gdf[weather_gdf['Location'] == loc].copy()

## Interpolation of Weather Stations Workflow...

In [ ]:
# read 2022 shapefile of the weather station locations  
tahmo_test = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geo_Interpolation/Notebook/tahmo_gdf_2022.shp')

### Reproject Tahmo to EPSG 32630 
tahmo_test = tahmo_test.to_crs('EPSG:32630')
tahmo_test = tahmo_test.rename(columns = {'Site_ID':'Location'})

### Plot Weather Stations and Enumeration Areas 

In [ ]:
## read the Greater Accra shapefile data & reproject to local CRS 
ea_shapefile = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geo_Interpolation/Notebook/GAMA_20200923_eaBorder.shp')
ea_shapefile = ea_shapefile.to_crs('EPSG:32630')


## read shapefile of just the EAs with PQR data 
eas_of_interest = gpd.read_file("/home/kdonkor_umass_edu/Interpolation_Geospatial_All_EAs/study_area_copy.geojson")

In [ ]:
fig, ax = plt.subplots(figsize = (18,15), dpi = 250)

# plot of EAs without data 
ea_shapefile.plot(ax = ax, alpha = 0.5, edgecolor = 'white', lw = 0.6, color = 'grey', label = 'Enumeration Areas')

# EAs of Interest 
eas_of_interest.plot(ax = ax, alpha = 0.5, facecolor = 'yellow', lw = 0.6, label = 'EAs of Interest')

# Tahmo Stations Plot 
tahmo_test.plot(color = 'red', ax = ax, marker='*', markersize = 12, legend = True, label = 'Weather Station')

# basemap 
ctx.add_basemap(ax, zoom='auto', source=ctx.providers.OpenStreetMap.Mapnik, attribution=None, crs=(ea_shapefile.crs))

# Define legend entries  
enumeration_area_patch = mpatches.Patch(color='grey', alpha=0.5, label="Enumeration Area")
tahmo_patch = mlines.Line2D([], [], color='red', marker='*', linestyle='None', markersize=5, label="Weather Station")

# Add legend  
ax.legend(handles=[enumeration_area_patch, tahmo_patch], loc='lower right', fontsize=14, handleheight=2, handlelength=4)

# plot title 
ax.set_title('Study Area', fontsize = 18, pad = 10)

plt.show()

### Define 6km radius around weather stations 

In [ ]:
buffer_radius = 6000

tahmo_buffers = tahmo_test.copy()
tahmo_buffers['geometry'] = tahmo_buffers.geometry.buffer(buffer_radius)

In [ ]:
# Create a figure and axis for plotting
fig, ax = plt.subplots(figsize=(18, 15), dpi=250)

# Plot Enumeration Areas (EA) without data
ea_shapefile.plot(ax=ax, alpha=0.5, edgecolor='white', lw=0.6, color='grey', label='Enumeration Areas')

# EAs of Interest 
eas_of_interest.plot(ax = ax, alpha = 0.5, facecolor = 'yellow', lw = 0.6, label = 'EAs of Interest')

# Plot Buffer circles (in semi-transparent color)
tahmo_buffers.plot(ax=ax, color='blue', alpha=0.3, label='5 km Buffer')

# Plot Tahmo Stations as red stars
tahmo_test.plot(color='red', ax=ax, marker='*', markersize=12, legend=True, label='Weather Station')

# Add Basemap
ctx.add_basemap(ax, zoom='auto', source=ctx.providers.OpenStreetMap.Mapnik, attribution=None, crs=(ea_shapefile.crs))

# Define legend entries  
enumeration_area_patch = mpatches.Patch(color='grey', alpha=0.5, label="Enumeration Area")
sensor_patch = mlines.Line2D([], [], color='red', marker='*', linestyle='None', markersize=5, label="Weather Station")
buffer_patch = mpatches.Patch(color='blue', alpha=0.3, label="Buffer")

# Add legend  
ax.legend(handles=[enumeration_area_patch, sensor_patch, buffer_patch], loc='lower right', fontsize=14, handleheight=2, handlelength=4)

# Plot title 
ax.set_title('Study Area', fontsize=18, pad=10)

# Show the plot
plt.show()


### EAs that intersect with weather station buffer 

In [ ]:
unioned_buffer = unary_union(tahmo_buffers['geometry'])

tahmo_buffer_union_gdf = gpd.GeoDataFrame(geometry=[unioned_buffer], crs=tahmo_buffers.crs)


## EAs that intersected with buffers 
eas_n_buffer = gpd.sjoin(eas_of_interest, tahmo_buffer_union_gdf, how = 'inner', predicate = 'intersects')

eas_n_buffer = eas_n_buffer.drop_duplicates(subset='ea_code9ch')
eas_n_buffer = eas_n_buffer[['ea_code9ch', 'geometry']]
eas_n_buffer = eas_n_buffer.reset_index(drop = True)

In [ ]:
# Convert list of ea codes to DataFrame
ea_df = pd.DataFrame(eas_n_buffer['ea_code9ch'].tolist(), columns=['ea_code9ch'])

# Save to CSV
# ea_df.to_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Weather/Hourly_Weather_per_EA/ea_code9ch_list_6km_buffer.csv', index=False)

In [ ]:
## Plot to Visualize 

# Create a figure and axis for plotting
fig, ax = plt.subplots(figsize=(14, 10), dpi=250)

# Plot Enumeration Areas (EA) without data
ea_shapefile.plot(ax=ax, alpha=0.5, edgecolor='white', lw=0.6, color='grey', label='Enumeration Areas')

# EAs (INTERSECTING with Buffer) 
eas_n_buffer.plot(ax = ax, alpha = 0.5, facecolor = 'green', lw = 0.6, label = 'EAs Intersecting with Buffer')

# Plot Buffer circles (in semi-transparent color)
tahmo_buffers.plot(ax=ax, color='blue', alpha=0.3, label='6 km Buffer')

# Plot Tahmo Stations as red stars
tahmo_test.plot(color='red', ax=ax, marker='*', markersize=12, legend=True, label='Weather Station')

# Add Basemap
ctx.add_basemap(ax, zoom='auto', source=ctx.providers.OpenStreetMap.Mapnik, attribution=None, crs=(ea_shapefile.crs))

# Define legend entries  
enumeration_area_patch = mpatches.Patch(color='grey', alpha=0.5, label="Enumeration Area")
sensor_patch = mlines.Line2D([], [], color='red', marker='*', linestyle='None', markersize=5, label="Weather Station")
buffer_patch = mpatches.Patch(color='blue', alpha=0.3, label="Buffer")

# Add legend  
ax.legend(handles=[enumeration_area_patch, sensor_patch, buffer_patch], loc='lower right', fontsize=14, handleheight=2, handlelength=4)

ax.set_xlim(790000,834000)

# Plot title 
ax.set_title('Study Area', fontsize=18, pad=10)

# Show the plot
plt.show()


### Retrieve new list of pqr sites from these EAs 

In [ ]:
intersecting_sites_gdf = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Weather/Hourly_Weather_per_EA/intersecting_sites.geojson')

intersecting_sites_gdf = intersecting_sites_gdf[['space_grouping', 'ea_code9ch', 'geometry']]
intersecting_sites_gdf = intersecting_sites_gdf.rename(columns = {'space_grouping':'site_id'})
intersecting_sites_gdf['ea_code9ch'] = intersecting_sites_gdf['ea_code9ch'].astype(int)


new_list_eas = gpd.sjoin(intersecting_sites_gdf, eas_n_buffer, how = 'inner', predicate = 'intersects')
new_list_eas = new_list_eas[['site_id', 'geometry']].reset_index(drop=True)
new_list_eas = new_list_eas.drop_duplicates(subset='site_id')

new_pqr_site_list = np.sort(new_list_eas.site_id.unique().tolist())

In [ ]:
# Convert list of PQR sites (within the 6km buffers) to DataFrame
df_sites_new = pd.DataFrame({'site_id': sorted(new_pqr_site_list)})


# Save to CSV
# df_sites_new.to_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/grid_weather_sites_list_6km_buffer.csv', index=False) 

### EAs and Weather Buffer matches 

In [ ]:
ea_with_station_matches = gpd.sjoin(eas_n_buffer, tahmo_buffers[['geometry', 'Location']], how = 'left', predicate = 'intersects')


ea_station_list = ea_with_station_matches.groupby(ea_with_station_matches.ea_code9ch)['Location'] \
                                         .apply(lambda x: list(set(x.dropna()))) \
                                         .reset_index(name='Intersecting_Stations')

### EAs and PQR site matches  

In [ ]:
ea_n_site_matches = gpd.sjoin(eas_n_buffer, intersecting_sites_gdf[['geometry', 'site_id']], how = 'left', predicate = 'intersects')

ea_site_list = ea_n_site_matches.groupby(ea_n_site_matches.ea_code9ch)['site_id'] \
                                         .apply(lambda x: list(set(x.dropna()))) \
                                         .reset_index(name='Intersecting_Sites')


# ea_site_list.to_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/ea_site_list_6km_buffer.csv', index=False) 

### Combined EAs and PQRs 

In [ ]:
merged_eas_sites_stations = pd.merge(ea_site_list, ea_station_list, on = 'ea_code9ch', how = 'left')

### Weather Station --> Gdf Map 

In [ ]:
station_df_map = {
    'St_Johns': st_johns_df_year,
    'Temasco': temasco_df_year,
    'G_Met': g_met_hq_df_year,
    'Accra_Aca': accra_aca_df_year,
    'Safisana': safisana_df_year,
    'Nsawam': nsawam_df_year,
    'Agri_Impact': agri_impact_df_year,
    'Accra_Girls': accra_girls_df_year,
    'Legon': legon_df_year,
    'Madina': madina_df_year,
    'Berekuso': berekuso_df_year
}

### IDW Interpolation   

In [ ]:
### Core interpolation function (embedded in the interpolation per hour function) 


def idw_interpolation(x, y, values, grid_x, grid_y, power, k=5):

    # Create KDTree for fast nearest-neighbor search
    tree = cKDTree(np.column_stack((x, y)))
    
    # Query the tree to find all neighbors (len(x))
    distances, indices = tree.query(np.column_stack((grid_x, grid_y)), k = min(k, len(x)))
    
    # Initialize the array for interpolated values
    interpolated_values = np.zeros(grid_x.shape)
    
    
    # Inside the loop
    for i in range(len(grid_x)):
        dist = np.atleast_1d(distances[i])
        vals = np.atleast_1d(values[indices[i]])

        # Filter out NaN values
        valid = ~np.isnan(vals)
        if not np.any(valid):
            interpolated_values[i] = np.nan
            continue

        dist = dist[valid]
        vals = vals[valid]

        dist[dist == 0] = 1e-10
        weights = 1 / (dist ** power)
        interpolated_values[i] = np.sum(weights * vals) / np.sum(weights)
    
    return interpolated_values

In [ ]:
### Run interpolation for each EA, per hour 


def perform_idw_interpolation_by_hour_EA(weather_station_gdf, value_col, ea_study_area, power, k):
    weather_station_gdf = weather_station_gdf.copy()

    # Ensure 'Hour' is datetime and set as index
    if weather_station_gdf.index.name != 'Hour':
        weather_station_gdf['Hour'] = pd.to_datetime(weather_station_gdf['Hour'])
        weather_station_gdf = weather_station_gdf.set_index('Hour')

    # Extract year and keep Hour string for naming
    weather_station_gdf['Year'] = weather_station_gdf.index.year
    weather_station_gdf['Hour_str'] = weather_station_gdf.index.strftime('%Y-%m-%d %H:00:00')

    # Unique hours from index
    unique_hours = weather_station_gdf.index.unique()
    interpolated_values_by_hour = {}

    for hour in unique_hours:
        hour_data = weather_station_gdf.loc[hour]
        if hour_data.empty:
            continue

        # Ensure hour_data is still a DataFrame (not a Series)
        if isinstance(hour_data, pd.Series):
            hour_data = hour_data.to_frame().T

            ##### 
            
        # Extract values and filter out NaNs
        x_all = hour_data.geometry.x.values
        y_all = hour_data.geometry.y.values
        values_all = hour_data[value_col].values

        # Filter for non-NaN values only
        valid_mask = ~np.isnan(values_all)
        x = x_all[valid_mask]
        y = y_all[valid_mask]
        values = values_all[valid_mask]

        
        # Skip if nothing left
        if len(values) == 0:
            interpolated_values = [np.nan] * len(ea_study_area)
            column_name = f'interp_val_{value_col}_Hour_{hour.strftime("%Y-%m-%d %H:00:00")}'
            interpolated_values_by_hour[column_name] = interpolated_values
            continue
        
            ##### 
            
        interpolated_values = []
        for _, cell in ea_study_area.iterrows():
            minx, miny, maxx, maxy = cell.geometry.bounds
            gridx = np.linspace(minx, maxx, 10)
            gridy = np.linspace(miny, maxy, 10)
            grid_x, grid_y = np.meshgrid(gridx, gridy)
            grid_x_flat = grid_x.flatten()
            grid_y_flat = grid_y.flatten()

            # Perform IDW interpolation
            interpolated_grid_values = idw_interpolation(x, y, values, grid_x_flat, grid_y_flat, power, k)
            avg_value = np.nanmean(interpolated_grid_values) if interpolated_grid_values.size > 0 else np.nan
            interpolated_values.append(avg_value)

        column_name = f'interp_val_{value_col}_Hour_{hour.strftime("%Y-%m-%d %H:00:00")}'
        interpolated_values_by_hour[column_name] = interpolated_values

    # Combine results into DataFrame
    interpolated_df = pd.DataFrame(interpolated_values_by_hour)
    ea_study_area = pd.concat([ea_study_area, interpolated_df], axis=1)

    # Melt into long format
    melted_gdf = ea_study_area.melt(
        id_vars=['ea_code9ch', 'geometry'],
        value_vars=interpolated_df.columns,
        var_name='time',
        value_name=f'interp_val_{value_col}_Hour'
    )

    # Extract datetime from column name
    melted_gdf['time'] = melted_gdf['time'].str.extract(r'Hour_(.*)')[0]
    melted_gdf['time'] = pd.to_datetime(melted_gdf['time'])

    # Reorder columns
    melted_gdf = melted_gdf[['time', 'ea_code9ch', 'geometry', f'interp_val_{value_col}_Hour']]

    return melted_gdf

## Run Interpolations - 2022 

In [ ]:
weather_gdf_2022 = weather_gdf.sort_values(by = 'Hour').set_index('Hour').loc['2022']

#### Temperature 

In [ ]:
# temp_idw_2022 = perform_idw_interpolation_by_hour_EA(weather_gdf_2022, 'Temp', eas_n_buffer, power = 3, k = 5)

# temp_idw_2022.to_file("temp_idw_2022.geojson", driver="GeoJSON")

#### Precipitation 

In [ ]:
# precip_idw_2022 = perform_idw_interpolation_by_hour_EA(weather_gdf_2022, 'Precip', eas_n_buffer, power = 3, k = 5)

# precip_idw_2022.to_file("precip_idw_2022.geojson", driver="GeoJSON")

## Run Interpolations - 2023 

In [ ]:
weather_gdf_2023 = weather_gdf.sort_values(by = 'Hour').set_index('Hour').loc['2023']

#### Temperature 

In [ ]:
temp_idw_2023 = perform_idw_interpolation_by_hour_EA(weather_gdf_2023, 'Temp', eas_n_buffer, power = 3, k = 5)

# temp_idw_2023.to_file("temp_idw_2023_6km_buffer.geojson", driver="GeoJSON")

#### Precip 

In [ ]:
precip_idw_2023 = perform_idw_interpolation_by_hour_EA(weather_gdf_2023, 'Precip', eas_n_buffer, power = 3, k = 5)

# precip_idw_2023.to_file("precip_idw_2023.geojson_6km_buffer", driver="GeoJSON")